# Odseki coverage: mbtiles ↔ CSV

Questions:
1. Do all odseki from `heatmap_past_data.csv` exist in `odseki_vector_map.mbtiles`?
2. Do all odseki from the mbtiles exist in the CSV?
3. Is there data for every segment for every month?

> Unique segment key: `(odsek_id, ggo)`. The CSV stores `ggo` as an integer; mbtiles stores it as `ggo_naziv` string.
> The mapping is defined in `GGO_CODE_TO_NAZIV` below.

In [21]:
import sqlite3, math, gzip, struct
from collections import defaultdict
from pathlib import Path
import pandas as pd

# ── paths (adjust if running from a different working directory)
BASE = Path("..").resolve()
MBTILES_PATH = BASE / "data" / "vector_map_odseki.mbtiles"
CSV_PATH     = BASE / "data" / "heatmap_past_data.csv"

# ── same mapping as run_web_server.py
GGO_CODE_TO_NAZIV = {
    1: 'TOLMIN',     2: 'BLED',          3: 'KRANJ',
    4: 'LJUBLJANA',  5: 'POSTOJNA',      6: 'KOČEVJE',
    7: 'NOVO MESTO', 8: 'BREŽICE',       9: 'CELJE',
   10: 'NAZARJE',   11: 'SLOVENJ GRADEC',12: 'MARIBOR',
   13: 'MURSKA SOBOTA', 14: 'SEŽANA',
}
GGO_NAZIV_TO_CODE = {v: k for k, v in GGO_CODE_TO_NAZIV.items()}

print(f"mbtiles: {MBTILES_PATH}")
print(f"CSV:     {CSV_PATH}")

mbtiles: /home/tiln/Documents/BarkWatch_Arnes-Hackathon-2026_interface/data/vector_map_odseki.mbtiles
CSV:     /home/tiln/Documents/BarkWatch_Arnes-Hackathon-2026_interface/data/heatmap_past_data.csv


## 1. Load CSV — unique segments and months

In [22]:
df = pd.read_csv(CSV_PATH)
print(f"Total rows: {len(df):,}")
print(f"Columns:    {list(df.columns)}\n")

# Drop rows with unknown ggo
unknown_ggo = df[~df["ggo"].isin(GGO_CODE_TO_NAZIV)]
if len(unknown_ggo):
    print(f"WARNING: {len(unknown_ggo):,} rows with unknown ggo codes: {sorted(unknown_ggo['ggo'].unique())}")
    df = df[df["ggo"].isin(GGO_CODE_TO_NAZIV)].copy()

# Normalise: add ggo_naziv so we can compare against mbtiles
df["ggo_naziv"] = df["ggo"].map(GGO_CODE_TO_NAZIV)

all_months_csv = sorted(df["leto_mesec"].unique())
print(f"Month range: {all_months_csv[0]} → {all_months_csv[-1]}  ({len(all_months_csv)} months)")

# Unique segments as (odsek_id, ggo_naziv)
csv_segments = set(zip(df["odsek_id"], df["ggo_naziv"]))
print(f"Unique segments (odsek_id, ggo_naziv): {len(csv_segments):,}")

Total rows: 310,481
Columns:    ['ggo', 'odsek_id', 'leto_mesec', 'target']

Month range: 2007-02 → 2025-12  (227 months)
Unique segments (odsek_id, ggo_naziv): 34,407


## 2. Extract all odseki from mbtiles

Decodes all tiles at the highest zoom that covers the whole country (zoom 11, same as the server's bbox index). Uses the minimal MVT decoder so no external dependencies are needed.

In [23]:
# ── Minimal MVT decoder (no external deps) ──────────────────────────────────

def _read_varint(data, pos):
    result = shift = 0
    while pos < len(data):
        b = data[pos]; pos += 1
        result |= (b & 0x7F) << shift
        if not (b & 0x80): return result, pos
        shift += 7
    return result, pos

def _zigzag(n): return (n >> 1) ^ -(n & 1)

def _unpack_varints(data):
    values, pos = [], 0
    while pos < len(data):
        v, pos = _read_varint(data, pos)
        values.append(v)
    return values

def _decode_value(data):
    pos = 0
    while pos < len(data):
        tw, pos = _read_varint(data, pos)
        fn, wt = tw >> 3, tw & 0x7
        if wt == 0:
            v, pos = _read_varint(data, pos)
            if fn in (4, 5): return v
            if fn == 6: return _zigzag(v)
            if fn == 7: return bool(v)
        elif wt == 2:
            l, pos = _read_varint(data, pos)
            chunk = data[pos:pos+l]; pos += l
            if fn == 1: return chunk.decode('utf-8', errors='replace')
        elif wt == 1: v = struct.unpack_from('<d', data, pos)[0]; pos += 8; return v if fn == 3 else None
        elif wt == 5: v = struct.unpack_from('<f', data, pos)[0]; pos += 4; return v if fn == 2 else None
    return None

def decode_tile_features(tile_bytes, target_layer='odsek'):
    """Yield property dicts for all features in target_layer."""
    if tile_bytes[:2] == b'\x1f\x8b':
        tile_bytes = gzip.decompress(tile_bytes)
    pos = 0
    while pos < len(tile_bytes):
        tw, pos = _read_varint(tile_bytes, pos)
        fn, wt = tw >> 3, tw & 0x7
        if wt == 2:
            l, pos = _read_varint(tile_bytes, pos)
            chunk = tile_bytes[pos:pos+l]; pos += l
            if fn == 3:  # layer
                _yield_layer(chunk, target_layer)
        elif wt == 0: _, pos = _read_varint(tile_bytes, pos)
        elif wt == 1: pos += 8
        elif wt == 5: pos += 4

def _yield_layer(data, target_layer):
    keys, raw_vals, raw_feats, name = [], [], [], ''
    pos = 0
    while pos < len(data):
        tw, pos = _read_varint(data, pos)
        fn, wt = tw >> 3, tw & 0x7
        if wt == 2:
            l, pos = _read_varint(data, pos)
            chunk = data[pos:pos+l]; pos += l
            if fn == 1: name = chunk.decode('utf-8', errors='replace')
            elif fn == 2: raw_feats.append(chunk)
            elif fn == 3: keys.append(chunk.decode('utf-8', errors='replace'))
            elif fn == 4: raw_vals.append(_decode_value(chunk))
        elif wt == 0: _, pos = _read_varint(data, pos)
        elif wt == 1: pos += 8
        elif wt == 5: pos += 4
    if name != target_layer:
        return
    for fd in raw_feats:
        tags_raw = None; fp = 0
        while fp < len(fd):
            tw, fp = _read_varint(fd, fp)
            fn2, wt2 = tw >> 3, tw & 0x7
            if wt2 == 0: _, fp = _read_varint(fd, fp)
            elif wt2 == 2:
                l, fp = _read_varint(fd, fp)
                chunk = fd[fp:fp+l]; fp += l
                if fn2 == 2: tags_raw = chunk
            elif wt2 == 1: fp += 8
            elif wt2 == 5: fp += 4
        if not tags_raw: continue
        tag_ints = _unpack_varints(tags_raw)
        props = {}
        for k in range(0, len(tag_ints)-1, 2):
            ki, vi = tag_ints[k], tag_ints[k+1]
            if ki < len(keys) and vi < len(raw_vals):
                props[keys[ki]] = raw_vals[vi]
        yield props

print("MVT decoder ready.")

MVT decoder ready.


In [24]:
DECODE_ZOOM = 11   # same zoom the server uses for the bbox cache
TARGET_LAYER = 'odseki_map_ggo_gge'

def extract_odsek_segments(mbtiles_path, zoom, target_layer=TARGET_LAYER):
    """Return set of (odsek_id, ggo_naziv) from all tiles at given zoom."""
    conn = sqlite3.connect(str(mbtiles_path))
    cur = conn.cursor()
    cur.execute("SELECT tile_data FROM tiles WHERE zoom_level=?", (zoom,))
    tile_rows = cur.fetchall()
    conn.close()
    print(f"  Tiles at zoom {zoom}: {len(tile_rows)}")
    print(f"  Reading layer: {target_layer}")

    segments = set()
    unknown_ggo = set()

    for (tile_data,) in tile_rows:
        if not tile_data:
            continue
        raw = bytes(tile_data)
        if raw[:2] == b'\x1f\x8b':
            raw = gzip.decompress(raw)

        # Walk top-level tile fields to find layer messages (field 3, wire type 2)
        pos = 0
        while pos < len(raw):
            tw, pos = _read_varint(raw, pos)
            fn, wt = tw >> 3, tw & 0x7
            if wt == 2:
                l, pos = _read_varint(raw, pos)
                chunk = raw[pos:pos+l]; pos += l
                if fn == 3:  # layer
                    for props in _yield_layer(chunk, target_layer):
                        ggo_n = str(props.get('ggo_naziv', '')).strip()
                        odsek = str(props.get('odsek',     '')).strip()
                        if not ggo_n or not odsek:
                            continue
                        if ggo_n not in GGO_NAZIV_TO_CODE:
                            unknown_ggo.add(ggo_n)
                            continue
                        segments.add((odsek, ggo_n))
            elif wt == 0: _, pos = _read_varint(raw, pos)
            elif wt == 1: pos += 8
            elif wt == 5: pos += 4

    if unknown_ggo:
        print(f"  WARNING: unknown ggo_naziv values (skipped): {sorted(unknown_ggo)}")
    return segments


mbtiles_segments = extract_odsek_segments(MBTILES_PATH, DECODE_ZOOM)
print(f"Unique segments in mbtiles (zoom {DECODE_ZOOM}): {len(mbtiles_segments):,}")

  Tiles at zoom 11: 154
  Reading layer: odseki_map_ggo_gge
Unique segments in mbtiles (zoom 11): 53,253


## 3. Cross-reference: CSV ↔ mbtiles

In [25]:
only_in_csv     = csv_segments - mbtiles_segments
only_in_mbtiles = mbtiles_segments - csv_segments
in_both         = csv_segments & mbtiles_segments

print(f"Segments in CSV:              {len(csv_segments):>7,}")
print(f"Segments in mbtiles:          {len(mbtiles_segments):>7,}")
print(f"In both:                      {len(in_both):>7,}")
print(f"Only in CSV (no tile):        {len(only_in_csv):>7,}")
print(f"Only in mbtiles (no CSV data):{len(only_in_mbtiles):>7,}")

Segments in CSV:               34,407
Segments in mbtiles:           53,253
In both:                       34,348
Only in CSV (no tile):             59
Only in mbtiles (no CSV data): 18,905


In [26]:
# Breakdown by GGO — how many segments are missing tiles, per GGO?
if only_in_csv:
    df_miss = pd.DataFrame(sorted(only_in_csv), columns=["odsek_id", "ggo_naziv"])
    print("=== Segments in CSV but NOT in mbtiles — by GGO ===")
    print(df_miss.groupby("ggo_naziv").size().sort_values(ascending=False).rename("missing_tile_count"))
    print(f"\nFirst 20 examples:")
    print(df_miss.head(20).to_string(index=False))
else:
    print("All CSV segments have a corresponding tile.")

=== Segments in CSV but NOT in mbtiles — by GGO ===
ggo_naziv
NAZARJE           36
SEŽANA            13
BREŽICE            6
MURSKA SOBOTA      2
SLOVENJ GRADEC     2
Name: missing_tile_count, dtype: int64

First 20 examples:
odsek_id     ggo_naziv
  02052B MURSKA SOBOTA
  03020B        SEŽANA
   03059        SEŽANA
   03070        SEŽANA
  03085A       NAZARJE
   03096        SEŽANA
  03109C       NAZARJE
   03114        SEŽANA
  03150D       NAZARJE
  03150E       NAZARJE
  03151D       NAZARJE
  03153A       NAZARJE
  03153B       NAZARJE
  03153C       NAZARJE
  03154E       NAZARJE
  03154F       NAZARJE
  03223C       NAZARJE
   04006        SEŽANA
  04011C        SEŽANA
  04012A        SEŽANA


In [27]:
if only_in_mbtiles:
    df_extra = pd.DataFrame(sorted(only_in_mbtiles), columns=["odsek_id", "ggo_naziv"])
    print("=== Segments in mbtiles but NOT in CSV — by GGO ===")
    print(df_extra.groupby("ggo_naziv").size().sort_values(ascending=False).rename("no_csv_data_count"))
    print(f"\nFirst 20 examples:")
    print(df_extra.head(20).to_string(index=False))
else:
    print("Every tile segment has CSV data.")

=== Segments in mbtiles but NOT in CSV — by GGO ===
ggo_naziv
MARIBOR           3240
CELJE             2933
SEŽANA            2339
MURSKA SOBOTA     1878
TOLMIN            1850
POSTOJNA          1210
KOČEVJE           1146
BREŽICE           1047
LJUBLJANA          835
NOVO MESTO         820
SLOVENJ GRADEC     436
KRANJ              431
BLED               374
NAZARJE            366
Name: no_csv_data_count, dtype: int64

First 20 examples:
odsek_id ggo_naziv
  01 58A      BLED
  01 83V      BLED
  01 85V      BLED
  01 86A      BLED
  01 86V      BLED
  01 87A      BLED
  01 87V      BLED
  01 88A      BLED
  01 88V      BLED
  01 89B      BLED
  01 92V      BLED
  01 92Z      BLED
  01 93A      BLED
  01 93B      BLED
  01 93Z      BLED
  01 94V      BLED
  01 95A      BLED
  01 95Z      BLED
  01 97Z      BLED
  01 99Z      BLED


## 4. Month coverage per segment

For every segment that exists in both sources, how many months have data? Are there gaps?

In [28]:
n_months = len(all_months_csv)

# Count distinct months per (odsek_id, ggo_naziv)
months_per_seg = (
    df.groupby(["odsek_id", "ggo_naziv"])["leto_mesec"]
    .nunique()
    .rename("n_months")
)

print(f"Total months in CSV: {n_months}")
print(f"Total segments with any data: {len(months_per_seg):,}\n")

# Distribution
dist = months_per_seg.value_counts().sort_index()
print("Distribution of month counts per segment (first 20 buckets):")
print(dist.head(20).to_string())

# How many segments have ALL months?
full_coverage = (months_per_seg == n_months).sum()
partial       = (months_per_seg < n_months).sum()
print(f"\nSegments with ALL {n_months} months:  {full_coverage:,}")
print(f"Segments with PARTIAL coverage:       {partial:,}")

Total months in CSV: 227
Total segments with any data: 34,407

Distribution of month counts per segment (first 20 buckets):
n_months
1     5817
2     3915
3     3009
4     2472
5     2050
6     1803
7     1606
8     1326
9     1208
10    1119
11     981
12     882
13     764
14     681
15     585
16     568
17     491
18     458
19     431
20     365

Segments with ALL 227 months:  0
Segments with PARTIAL coverage:       34,407


In [29]:
# Segments with the fewest months — likely new or retired segments
sparse = months_per_seg[months_per_seg < n_months].reset_index().sort_values("n_months")
print(f"Segments with fewer than {n_months} months — sample (first 30):")
print(sparse.head(30).to_string(index=False))

# Check: which months are they missing?
print("\n--- Missing months for 5 most sparse segments ---")
all_months_set = set(all_months_csv)
for _, row in sparse.head(5).iterrows():
    seg_months = set(df[(df["odsek_id"] == row["odsek_id"]) &
                        (df["ggo_naziv"] == row["ggo_naziv"])]["leto_mesec"])
    missing = sorted(all_months_set - seg_months)
    print(f"  {row['odsek_id']} / {row['ggo_naziv']}: {row['n_months']} months, "
          f"missing {len(missing)} (e.g. {missing[:5]})")

Segments with fewer than 227 months — sample (first 30):
odsek_id      ggo_naziv  n_months
   90K13      LJUBLJANA         1
  93B10B      LJUBLJANA         1
   93B06      LJUBLJANA         1
   93B05      LJUBLJANA         1
   93B02      LJUBLJANA         1
   93A06      LJUBLJANA         1
   93F04      LJUBLJANA         1
  93E01A      LJUBLJANA         1
  93C17B      LJUBLJANA         1
  22065E        MARIBOR         1
   05010     NOVO MESTO         1
  05009B          KRANJ         1
  05009A         TOLMIN         1
  05009A  MURSKA SOBOTA         1
  22073B        MARIBOR         1
  22072C        MARIBOR         1
  08254B SLOVENJ GRADEC         1
   08D05       POSTOJNA         1
   08C02       POSTOJNA         1
   90K11      LJUBLJANA         1
  90K09A      LJUBLJANA         1
  90J32A      LJUBLJANA         1
  90J21B      LJUBLJANA         1
  01 88B           BLED         1
  01 83A           BLED         1
  01 81V           BLED         1
   90L12      LJUBLJANA  

In [30]:
# Per-month: how many segments have data in each month?
segs_per_month = df.groupby("leto_mesec")[["odsek_id","ggo_naziv"]].apply(
    lambda g: len(set(zip(g["odsek_id"], g["ggo_naziv"])))
).rename("n_segments")

print("Segments with data per month (min / max / mean):")
print(f"  min:  {segs_per_month.min():,}  ({segs_per_month.idxmin()})")
print(f"  max:  {segs_per_month.max():,}  ({segs_per_month.idxmax()})")
print(f"  mean: {segs_per_month.mean():,.0f}")

print(f"\nMonths with fewer than {segs_per_month.max()} segments:")
low_months = segs_per_month[segs_per_month < segs_per_month.max()]
print(low_months.sort_values().to_string() if len(low_months) else "  None — all months have full coverage.")

Segments with data per month (min / max / mean):
  min:  66  (2010-02)
  max:  4,412  (2022-09)
  mean: 1,368

Months with fewer than 4412 segments:
leto_mesec
2010-02      66
2009-02      77
2013-02      95
2011-02     128
2014-02     163
2010-01     169
2009-01     170
2013-01     175
2008-01     185
2011-01     190
2022-01     196
2012-02     204
2013-03     206
2021-02     208
2010-03     225
2022-02     247
2021-01     249
2015-02     296
2012-01     314
2015-01     334
2013-04     347
2018-02     366
2010-04     391
2009-03     398
2022-04     411
2011-03     429
2018-01     450
2019-02     450
2023-01     452
2008-02     466
2009-04     499
2011-04     504
2024-01     519
2021-04     522
2012-03     529
2022-05     539
2009-05     555
2020-01     559
2025-01     579
2013-06     626
2021-06     631
2009-07     634
2012-06     641
2011-07     643
2021-05     649
2021-03     650
2024-02     651
2014-01     655
2011-05     656
2020-02     662
2012-04     663
2016-01     669
2010-07 